# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

# Punctuation subtype ablation on identical prompt-grouped folds.
texts=train['text'].fillna('').astype(str).to_numpy()
labels=train['model'].astype(str).to_numpy()
groups=train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

punct_groups={
    'comma_colon_semicolon':[' ',',',';',':'],
    'brackets_quotes':['(',')','[',']','{','}','"',"'"],
    'dash_slash':['-','/'],
    'question_exclamation':['?','!'],
    'dot_ellipsis':['.'],
}

def punct_features(text, chars):
    n=max(len(text),1)
    return [text.count(c)/n for c in chars]

def make_text_features():
    return FeatureUnion([
        ('word',TfidfVectorizer(analyzer='word',ngram_range=(1,3),min_df=2,sublinear_tf=True,max_features=120000)),
        ('char',TfidfVectorizer(analyzer='char',ngram_range=(2,5),min_df=2,sublinear_tf=True,max_features=120000)),
    ])

configs={'word_char':None}
for name,chars in punct_groups.items(): configs['word_char_'+name]=chars
configs['word_char_all_punctuation']=sum(punct_groups.values(),[])
fold_indices=list(GroupKFold(n_splits=5).split(texts,labels,groups=groups))
results={}
for config,chars in configs.items():
    scores=[]
    for fold,(tr,va) in enumerate(fold_indices,1):
        tf=make_text_features(); Xtr_text=tf.fit_transform(texts[tr]); Xva_text=tf.transform(texts[va])
        Xtr,Xva=Xtr_text,Xva_text
        if chars is not None:
            tr_style=np.asarray([punct_features(texts[i],chars) for i in tr],dtype=np.float32)
            va_style=np.asarray([punct_features(texts[i],chars) for i in va],dtype=np.float32)
            scaler=StandardScaler(); Xtr=hstack([Xtr,csr_matrix(scaler.fit_transform(tr_style))],format='csr'); Xva=hstack([Xva,csr_matrix(scaler.transform(va_style))],format='csr')
        model=LinearSVC(C=1.0,class_weight='balanced',max_iter=1200); model.fit(Xtr,labels[tr])
        score=float(balanced_accuracy_score(labels[va],model.predict(Xva))); scores.append(score)
        print(f'{config} fold {fold}: {score:.4f}')
    results[config]={'mean_balanced_accuracy':float(np.mean(scores)),'std_balanced_accuracy':float(np.std(scores,ddof=1)),'fold_scores':scores}
    print(f'CONFIG {config}: mean={np.mean(scores):.4f}')
ranked=sorted(results.items(),key=lambda kv:kv[1]['mean_balanced_accuracy'],reverse=True)
summary={'experiment':'punctuation subtype ablation','split_method':'5-fold GroupKFold by prompt_id','configs':results,'ranking':[{'config':k,**v} for k,v in ranked]}
print('\\nSUMMARY:',json.dumps(summary,indent=2)); Path('/kaggle/working/punctuation_results.json').write_text(json.dumps(summary,indent=2)); print('Saved:', '/kaggle/working/punctuation_results.json')


## Punctuation subtype ablation

This run keeps word/character TF-IDF fixed and compares punctuation groups under identical GroupKFold splits.
